# Notebook 7: LLM-as-a-Judge Evaluation

Notebook này sử dụng LLM (ví dụ: Google Gemini API hoặc Groq/Llama-3) làm trọng tài để chấm điểm các bản tóm tắt được sinh ra từ các mô hình (BARTpho Full FT, BARTpho LoRA, Qwen LoRA).

Các tiêu chí (Thang 1-5):
1. Relevance (Sự liên quan)
2. Coherence (Tính mạch lạc)
3. Consistency (Tính nhất quán / Faithfulness)
4. Fluency (Tính trôi chảy)

In [ ]:
!pip install -q google-genai pandas tqdm matplotlib seaborn

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from google import genai
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình API Key (Thay bằng API Key của bạn trên Kaggle Secrets)
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# GOOGLE_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=GOOGLE_API_KEY)

# Sử dụng mô hình Gemini 1.5 Flash (nhanh, rẻ, đủ tốt cho NLP Evaluation)
model_id = 'gemini-3.5-flash-lite'

## 1. Load Dữ liệu
Giả sử bạn đã chạy Notebook 5 và lưu kết quả sinh (generated summaries) của các mô hình vào một file CSV, ví dụ `all_models_predictions.csv`.
Ở đây ta tạo mock data nếu chưa có.

In [ ]:
try:
    df = pd.read_csv("all_models_predictions.csv")
    # Lấy 50 mẫu ngẫu nhiên để tiết kiệm API và thời gian
    df_sample = df.sample(50, random_state=42).reset_index(drop=True)
except FileNotFoundError:
    print("Chưa tìm thấy file predictions, vui lòng tạo từ Notebook 5. Sử dụng mock data để test code...")
    df_sample = pd.DataFrame({
        "article": ["Bài báo gốc 1..."] * 3,
        "abstract": ["Tóm tắt chuẩn 1..."] * 3,
        "bartpho_full": ["Tóm tắt sinh bởi BARTpho Full..."] * 3,
        "bartpho_lora": ["Tóm tắt sinh bởi BARTpho LoRA..."] * 3,
        "qwen_lora": ["Tóm tắt sinh bởi Qwen LoRA..."] * 3
    })

## 2. Khởi tạo Prompt cho LLM Judge

In [ ]:
prompt_template = """
Bạn sẽ được cung cấp một bài báo gốc và một bản tóm tắt của nó. Nhiệm vụ của bạn là đánh giá bản tóm tắt này dựa trên 4 tiêu chí khác nhau.
Vui lòng đọc và hiểu kỹ các hướng dẫn này.

Bài báo gốc: {article}
Bản tóm tắt cần đánh giá: {summary}

Tiêu chí đánh giá (Thang điểm từ 1 đến 5, trong đó 1 là Rất kém, 5 là Rất tốt):

I. Relevance (Sự liên quan)
Bản tóm tắt chỉ nên bao gồm những thông tin quan trọng. Có bao quát các ý chính không? Có bị dư thừa không?

II. Coherence (Tính mạch lạc)
Bản tóm tắt phải được cấu trúc tốt, liên kết logic giữa các câu, không rời rạc.

III. Consistency (Tính nhất quán / Faithfulness)
Bản tóm tắt chỉ chứa các khẳng định suy ra từ tài liệu gốc. Không bịa đặt (hallucinate), sai sự thật.

IV. Fluency (Tính trôi chảy)
Đánh giá ngữ pháp, chính tả, cách chọn từ, hành văn tự nhiên.

Vui lòng CHỈ trả về kết quả dưới định dạng JSON sau, không giải thích thêm:
{{"Relevance": <điểm 1-5>, "Coherence": <điểm 1-5>, "Consistency": <điểm 1-5>, "Fluency": <điểm 1-5>}}
"""

## 3. Hàm chấm điểm qua API

In [ ]:
def evaluate_summary(article, summary):
    prompt = prompt_template.format(article=article, summary=summary)
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model_id, contents=prompt)
            text = response.text
            # Làm sạch chuỗi JSON nếu LLM trả về dư markdown codeblock
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0].strip()
            elif "```" in text:
                text = text.split("```")[1].strip()
                
            return json.loads(text)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Lỗi API hoặc Parse JSON: {e}")
                return {"Relevance": None, "Coherence": None, "Consistency": None, "Fluency": None}
            time.sleep(30) # Bị rate limit, đợi 30s rồi thử lại

## 4. Đánh giá tự động
Lưu ý: Code dưới đây sẽ gọi API cho từng mô hình trên tập sample.

In [ ]:
results = []
models_to_eval = ["bartpho_full_beam", "bartpho_lora_beam", "qwen_lora_beam"]

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    article = row["article"]
    
    for model_col in models_to_eval:
        summary = row.get(model_col)
        if not summary:
            continue
            
        scores = evaluate_summary(article, summary)
        
        results.append({
            "id": idx,
            "model": model_col,
            "Relevance": scores["Relevance"],
            "Coherence": scores["Coherence"],
            "Consistency": scores["Consistency"],
            "Fluency": scores["Fluency"]
        })
        time.sleep(4) # Tránh rate limit (15 RPM free tier)

results_df = pd.DataFrame(results)
results_df.to_csv("llm_judge_scores.csv", index=False)
results_df.head()

## 5. Phân tích & Trực quan hóa

In [ ]:
# Tính điểm trung bình của từng mô hình
mean_scores = results_df.groupby("model").mean().reset_index()
print(mean_scores)

# Vẽ Radar Chart (Biểu đồ mạng nhện)
def plot_radar_chart(df, categories):
    import numpy as np
    from math import pi
    
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    for idx, row in df.iterrows():
        values = row[categories].tolist()
        values += values[:1]
        
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=row['model'])
        ax.fill(angles, values, alpha=0.1)
        
    plt.xticks(angles[:-1], categories)
    ax.set_ylim(0, 5)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.title("LLM-as-a-Judge Radar Chart (1-5 Scale)", size=15, y=1.1)
    plt.show()

categories = ["Relevance", "Coherence", "Consistency", "Fluency"]
if not mean_scores.empty:
    plot_radar_chart(mean_scores, categories)